# 03. Multi-band photometry in one call

Measure a 50-arcsec Euclid field with `run_forced_photometry`. The driver
loads the catalog, images and PSFs, fits VIS profiles, then fits their fluxes
in Y, J and H. It returns measurements and diagnostic metadata.

[Notebook 01](01_multiband_forced_photometry.ipynb) explains the individual
steps and their limits. The source morphology is shared across bands;
estimated errors do not include every source of model or blend uncertainty.


In [1]:
# Locate the package when running from either notebook directory.
import os, sys
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "src/euclid_phot").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from the repository.")
if "euclid_phot" in sys.modules:
    loaded_from = Path(sys.modules["euclid_phot"].__file__).resolve()
    if loaded_from != (PROJECT_ROOT / "src/euclid_phot/__init__.py").resolve():
        raise RuntimeError("Restart the kernel to load this repository's package.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ.setdefault("EUCLID_PHOT_DATA_DIR", str(PROJECT_ROOT / "examples/native_data"))

import numpy as np
import euclid_phot as ep

DATA_DIR = Path(os.environ["EUCLID_PHOT_DATA_DIR"])
RA, DEC, SIZE_ARCSEC = 269.48, 67.30, 50.0
WISE_BANDS = ()  # optionally ("W1", "W2"); requires the WISE dependencies


## Run the field

`"tree"` selects and fits profiles on VIS; `"prior"` starts from MER
classifications and shapes. GRID supplies the nearest PSF sample for each
source inside the joint fit. Cutouts remain on the delivered MER pixel grid.

The first call for a new field downloads its inputs. Later calls reuse the
cache. A field crossing MER tile boundaries needs separate tile fits.


In [2]:
result = ep.run_forced_photometry(
    RA, DEC, SIZE_ARCSEC,
    prior={"band": "VIS", "objects": "mer", "model_selection": "tree"},
    target_bands={"euclid": ("Y", "J", "H"), "wise": WISE_BANDS},
    psf_product="grid", persource_psf=True,
    mask_bright_stars=True, with_flag=True, calibrate_errors=True,
    data_dir=DATA_DIR, n_workers=4, verbose=True,
)


[1/7] MER catalog from cache (mer_catalog_269.4800_67.3000_50.fits)
[2/7] discover mosaics for VIS, Y, J, H (lazy; only on a cache miss)
[3/7] fetch cutouts: VIS, Y, J, H
[4/7] build per-band PSFs (GRID-PSF, radius=60")
[5/7] build Tractor image + source models on VIS
[5a] STARSIGNAL bright-star mask: 2758 pixels (1.10%) excluded from the VIS fit


[6/7] VIS fit (free, forced-photometry)


[7/7] NISP forced phot: Y, J, H


[7a] calibrate flux errors (empty-position point-source fits)
  VIS: error inflation x1.00 (empty-position point-source, n=200)
  Y: error inflation x1.00 (empty-position point-source, n=200)
  J: error inflation x1.00 (empty-position point-source, n=200)
  H: error inflation x1.08 (empty-position point-source, n=200)


## Inspect the catalog

Flux densities are in microJansky. AB magnitudes are given for positive
fluxes; a flux the fit could not constrain, or that failed the flux guard,
is NaN. `reliable` is False when the source is near a bright star, on masked
pixels, or at the cutout edge.

The metadata records PSF treatment, uncertainty definitions and calibration
factors. Extinction-corrected magnitudes use the available foreground dust
estimate; the measured fluxes remain unchanged.


In [3]:
catalog = result.to_table()
print(f"{len(catalog)} sources; {len(catalog.colnames)} columns")
print("Euclid error inflation:", catalog.meta.get("error_inflation", {}))
print("PSF treatment:", catalog.meta["psf_mode"])
columns = ["object_id", "model", "flux_VIS_ujy", "flux_Y_ujy",
           "flux_J_ujy", "flux_H_ujy", "reliable"]
columns += [f"flux_{band}_ujy" for band in WISE_BANDS]
rows = np.flatnonzero(np.isfinite(catalog["flux_VIS_ujy"]))[:8]
catalog[columns][rows].pprint_all()


63 sources; 41 columns
Euclid error inflation: {'VIS': 1.0, 'Y': 1.0, 'J': 1.0, 'H': 1.082}
PSF treatment: nearest-source-position
     object_id         model        flux_VIS_ujy        flux_Y_ujy         flux_J_ujy         flux_H_ujy     reliable
                                        uJy                uJy                uJy                uJy                 
------------------- ------------ ------------------ ------------------ ------------------ ------------------ --------
2694794725673043335  PointSource 12.574807151989825  38.20572352120359  43.51588885486453  42.22385887385012     True
2694666079673060869  PointSource 5.9496773978599835 11.089766153804828 12.223123840099124 12.713989546223921     True
2694744461673017344 SersicGalaxy 10.982733815432555   25.4147582195474  31.87622461847772 39.025793731443116     True
2694874246673061089 SersicGalaxy 25.457240160570937  46.77857410138121 56.227718347782265  65.94376249104518    False
2694853134673020528    ExpGalaxy  3.5218912

Save the table with `catalog.write("photometry.csv", overwrite=True)`, or
as `.ecsv` to keep the units and metadata. For another field,
change `RA`, `DEC` and `SIZE_ARCSEC`, then rerun the call and catalog cells.
